# 10 Random forest tuning

**Goal:** improve the random forest chosen in 09: find at least 90% of the bumps on unseen roads with as few false alarms as possible.

**Rules:**
- One change at a time, always compared with the baseline (the RF from 05/09).
- Always the road split (leave one road out) and the bump-level code from 09.
- Every experiment goes into the log table, also the ones that do not help.

**Scores per experiment:**
- `avg_f1`, `avg_auc`: window level, average of the 3 test roads (as in 05)
- `found_default`, `fa_km_default`: bumps found (of 48) and real false alarms per km at threshold 0.5
- `fa_km_at_90`: the fewest false alarms per km while still finding at least 90% of the bumps (as in 09; threshold chosen on the test results, so only for comparing experiments)
- `spots_km_default`, `spots_km_at_90`: false alarm **spots** per km: neighbouring false alarm windows count as one spot (one place to check on the road). Added after experiment 1, because neighbour features make alarms come in groups.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

all_windows = pd.read_csv("data_features/windows_5m.csv")

# True for windows with a clear label (bump / no bump), False for "left out"
kept = all_windows["label"] != "left out"

# Relative features: std divided by the drive's calm level (as in 05)
std_features = ["GyrY_std", "AccZ_std", "GyrX_std", "GyrZ_std", "AccY_std", "AccX_std", "Pitch_std"]
for col in std_features:
    calm_level = all_windows[kept].groupby("drive")[col].quantile(0.25)
    all_windows[col + "_rel"] = all_windows[col] / all_windows["drive"].map(calm_level)

rel_features = [col + "_rel" for col in std_features]
print(all_windows.shape)

(2770, 52)


In [2]:
def bump_level(p):
    bumps_total = 0
    bumps_found = 0
    real_false_alarms = 0
    false_alarm_spots = 0

    for drive in p["drive"].unique():
        # This drive's windows, in road order.
        d = p[p["drive"] == drive].sort_values("window")

        # --- 1. Group the bump windows into bump passes ---
        bump_windows = d[d["is_bump"] == 1]["window"].tolist()
        bumps = []
        for w in bump_windows:
            if len(bumps) > 0 and w - bumps[-1][-1] == 1:
                bumps[-1].append(w)     # next to the previous bump window -> same bump
            else:
                bumps.append([w])       # a gap -> a new bump starts

        # --- 2. Was each bump found? ---
        flagged = d[d["pred"] == 1]["window"].tolist()
        for bump in bumps:
            bumps_total = bumps_total + 1
            for w in bump:
                if w in flagged:
                    bumps_found = bumps_found + 1
                    break

        # --- 3. Real false alarms ---
        alarms = d[(d["pred"] == 1) & (d["is_bump"] == 0)]["window"].tolist()
        real_fa_windows = []
        for a in alarms:
            near_a_bump = False
            for b in bump_windows:
                if abs(a - b) <= 2:
                    near_a_bump = True
            if not near_a_bump:
                real_false_alarms = real_false_alarms + 1
                real_fa_windows.append(a)

        # --- 4. Group neighbouring false alarm windows into spots ---
        for i in range(len(real_fa_windows)):
            if i == 0 or real_fa_windows[i] - real_fa_windows[i - 1] > 1:
                false_alarm_spots = false_alarm_spots + 1

    km = len(p) * 5 / 1000
    return {
        "bumps_total": bumps_total,
        "bumps_found": bumps_found,
        "detection_rate": bumps_found / bumps_total,
        "real_false_alarms": real_false_alarms,
        "false_alarms_per_km": real_false_alarms / km,
        "spots_per_km": false_alarm_spots / km,
    }

In [3]:
def run_experiment(name, features, data, seed=101, leaf=1):
    f1s = []
    aucs = []
    all_predictions = []

    # ---- Road split, as in 05 ----
    for test_road in data["road"].unique():
        train = data[data["road"] != test_road]
        test = data[data["road"] == test_road]

        model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed, min_samples_leaf=leaf)
        model.fit(train[features], train["is_bump"])

        prob = model.predict_proba(test[features])[:, 1]
        pred = (prob >= 0.5).astype(int)

        f1s.append(f1_score(test["is_bump"], pred))
        aucs.append(roc_auc_score(test["is_bump"], prob))

        p = test[["drive", "road", "window", "is_bump"]].copy()
        p["pred"] = pred
        p["prob"] = prob
        all_predictions.append(p)

    p = pd.concat(all_predictions)

    # ---- Bump level at the default threshold 0.5 ----
    default = bump_level(p)

    # ---- Bump level: fewest false alarm SPOTS while finding at least 90% ----
    best_spots = None
    best_fa = None
    best_t = None
    for t in np.arange(0.05, 1.0, 0.05):
        p["pred"] = (p["prob"] >= t).astype(int)
        b = bump_level(p)
        if b["detection_rate"] >= 0.9:
            if best_spots is None or b["spots_per_km"] < best_spots:
                best_spots = b["spots_per_km"]
                best_fa = b["false_alarms_per_km"]
                best_t = t

    return {
        "experiment": name,
        "avg_f1": np.mean(f1s),
        "avg_auc": np.mean(aucs),
        "found_default": default["bumps_found"],
        "fa_km_default": default["false_alarms_per_km"],
        "spots_km_default": default["spots_per_km"],
        "fa_km_at_90": best_fa,
        "spots_km_at_90": best_spots,
        "threshold_at_90": best_t,
    }

In [4]:
windows = all_windows[kept].copy()
windows["is_bump"] = (windows["label"] == "bump").astype(int)

log = []
log.append(run_experiment("baseline (05)", rel_features, windows))
print(pd.DataFrame(log).round(2).to_string(index=False))

   experiment  avg_f1  avg_auc  found_default  fa_km_default  spots_km_default  fa_km_at_90  spots_km_at_90  threshold_at_90
baseline (05)    0.44     0.76             46          13.42              9.16         6.74            4.84              0.6


### Experiment 1: neighbouring windows

Idea (from Jeremy): each window is judged without knowing its neighbours, so the calm flat top of a long heave looks like normal road. Here every window also gets the relative features of the window before (`_prev`) and after (`_next`) it.

#### Experiment 1a: neighbouring windows (±1 windows = 5 m each way)

In [5]:
# Add the previous and next window's features to each window, to give the model a bit of context.
for shift, name in [(1, "_prev"), (-1, "_next")]:
    other = all_windows[["drive", "window"] + rel_features].copy()
    other["window"] = other["window"] + shift
    other = other.rename(columns={col: col + name for col in rel_features})
    all_windows = all_windows.merge(other, on=["drive", "window"], how="left")

# A window at the start or end of a drive has no neighbour there: use its own value instead.
for col in rel_features:
    for name in ["_prev", "_next"]:
        all_windows[col + name] = all_windows[col + name].fillna(all_windows[col])

neighbour_features = rel_features + [col + "_prev" for col in rel_features] + [col + "_next" for col in rel_features]
print(len(neighbour_features), "features")

21 features


#### Experiment 1b: a wider view (±2 windows = 10 m each way)

In [6]:
# Neighbours two windows away (10 m before and after)
for shift, name in [(2, "_prev2"), (-2, "_next2")]:
    other = all_windows[["drive", "window"] + rel_features].copy()
    other["window"] = other["window"] + shift
    other = other.rename(columns={col: col + name for col in rel_features})
    all_windows = all_windows.merge(other, on=["drive", "window"], how="left")

for col in rel_features:
    for name in ["_prev2", "_next2"]:
        all_windows[col + name] = all_windows[col + name].fillna(all_windows[col])

neighbour2_features = neighbour_features + [col + "_prev2" for col in rel_features] + [col + "_next2" for col in rel_features]
print(len(neighbour2_features), "features")

35 features


#### Experiment 1c: ±3 windows (15 m each way)

In [7]:
# Neighbours three windows away (15 m before and after)
for shift, name in [(3, "_prev3"), (-3, "_next3")]:
    other = all_windows[["drive", "window"] + rel_features].copy()
    other["window"] = other["window"] + shift
    other = other.rename(columns={col: col + name for col in rel_features})
    all_windows = all_windows.merge(other, on=["drive", "window"], how="left")

for col in rel_features:
    for name in ["_prev3", "_next3"]:
        all_windows[col + name] = all_windows[col + name].fillna(all_windows[col])

neighbour3_features = neighbour2_features + [col + "_prev3" for col in rel_features] + [col + "_next3" for col in rel_features]
print(len(neighbour3_features), "features")

49 features


In [8]:
windows = all_windows[kept].copy()
windows["is_bump"] = (windows["label"] == "bump").astype(int)

log.append(run_experiment("1: neighbours ±1 window", neighbour_features, windows))
log.append(run_experiment("1b: neighbours ±2 windows", neighbour2_features, windows))
log.append(run_experiment("1c: neighbours ±3 windows", neighbour3_features, windows))
print(pd.DataFrame(log).round(2).to_string(index=False))

               experiment  avg_f1  avg_auc  found_default  fa_km_default  spots_km_default  fa_km_at_90  spots_km_at_90  threshold_at_90
            baseline (05)    0.44     0.76             46          13.42              9.16         6.74            4.84             0.60
  1: neighbours ±1 window    0.49     0.82             45          11.22              4.99        11.22            4.99             0.50
1b: neighbours ±2 windows    0.51     0.86             46          10.63              3.67        10.63            3.67             0.50
1c: neighbours ±3 windows    0.52     0.88             42           9.68              3.15        13.12            3.89             0.45


**Result of experiment 1:** with neighbours ±1 window, window F1 0.44 → 0.49 and AUC 0.76 → 0.82 (best so far). At the default threshold, false alarm spots per km almost halved (9.16 → 4.99) with 45 instead of 46 bumps found. At ≥ 90% found, both versions are equal (4.84 vs 4.99 spots/km).

Counting false alarm *windows* made the neighbour model look worse at ≥ 90% (11.22 vs 6.74 per km), because its alarms come in groups of neighbouring windows. Counted as spots (places to check on the road), it is equal.

**Decision:** keep the neighbour features. They are the new baseline.

**Result of experiment 1b:** with neighbours ±2 windows (25 m of context), every score improved again: window F1 0.51, AUC 0.86, 46 bumps found, false alarm spots at ≥ 90% found 3.67 per km (baseline 4.84, about 25% fewer).

Note: only the neighbours' sensor features are used, never their labels, so this is not leakage. In real use the whole drive is recorded, so the neighbouring windows are always available.

**Result of experiment 1c:** with neighbours ±3 windows (35 m of context), window F1 (0.52) and AUC (0.88) rose further, but only 42 of 48 bumps were found at the default threshold (88%, below the goal), and at ≥ 90% found the false alarm spots were higher (3.89 vs 3.67 per km). Too much context "averages away" short bumps.

**Decision:** neighbours ±2 windows (experiment 1b) is the best amount of context. Window-level scores alone would have picked ±3; the bump-level scores show it misses short bumps.

### Experiment 2: the heave’s tilt

Idea: so far the RF only knows how much the car shakes (std). But a heave also tilts the car: nose up as it drives onto the heave, nose down as it leaves. Pitch_mean (the car’s average tilt in the window) is already in the windows file, but we’ve never used it.

In [9]:
# same code as above, but in a function
def add_neighbours(df, cols, shift, name):
    other = df[["drive", "window"] + cols].copy()
    other["window"] = other["window"] + shift
    other = other.rename(columns={col: col + name for col in cols})
    df = df.merge(other, on=["drive", "window"], how="left")
    for col in cols:
        df[col + name] = df[col + name].fillna(df[col])
    return df

In [10]:
# Tilt compared to this drive's typical tilt
typical_tilt = all_windows[kept].groupby("drive")["Pitch_mean"].median()
all_windows["Pitch_tilt"] = all_windows["Pitch_mean"] - all_windows["drive"].map(typical_tilt)

# Its neighbours, ±2 windows (like the best context from experiment 1b)
for shift, name in [(1, "_prev"), (-1, "_next"), (2, "_prev2"), (-2, "_next2")]:
    all_windows = add_neighbours(all_windows, ["Pitch_tilt"], shift, name)

tilt_features = ["Pitch_tilt", "Pitch_tilt_prev", "Pitch_tilt_next", "Pitch_tilt_prev2", "Pitch_tilt_next2"]

In [11]:
windows = all_windows[kept].copy()
windows["is_bump"] = (windows["label"] == "bump").astype(int)

log.append(run_experiment("2: ±2 + pitch tilt", neighbour2_features + tilt_features, windows))
print(pd.DataFrame(log).round(2).to_string(index=False))

               experiment  avg_f1  avg_auc  found_default  fa_km_default  spots_km_default  fa_km_at_90  spots_km_at_90  threshold_at_90
            baseline (05)    0.44     0.76             46          13.42              9.16         6.74            4.84             0.60
  1: neighbours ±1 window    0.49     0.82             45          11.22              4.99        11.22            4.99             0.50
1b: neighbours ±2 windows    0.51     0.86             46          10.63              3.67        10.63            3.67             0.50
1c: neighbours ±3 windows    0.52     0.88             42           9.68              3.15        13.12            3.89             0.45
       2: ±2 + pitch tilt    0.52     0.85             44           9.09              3.30         9.09            3.30             0.50


### Is experiment 2 really better than 1b? Seed check

The difference is small (2 bumps vs ~5 false alarm spots), so we repeat both with 3 different random seeds.

In [12]:
seed_rows = []
for seed in [1, 2, 3]:
    seed_rows.append(run_experiment("1b: ±2", neighbour2_features, windows, seed))
    seed_rows.append(run_experiment("2: ±2 + tilt", neighbour2_features + tilt_features, windows, seed))

seed_table = pd.DataFrame(seed_rows)
print(seed_table[["experiment", "found_default", "spots_km_default", "spots_km_at_90"]].round(2).to_string(index=False))
print()
print(seed_table.groupby("experiment")[["avg_f1", "avg_auc", "found_default", "spots_km_at_90"]].mean().round(2))

  experiment  found_default  spots_km_default  spots_km_at_90
      1b: ±2             44              3.74            3.74
2: ±2 + tilt             44              3.37            3.37
      1b: ±2             45              3.30            3.30
2: ±2 + tilt             44              2.93            2.93
      1b: ±2             43              3.52            4.77
2: ±2 + tilt             45              3.37            3.37

              avg_f1  avg_auc  found_default  spots_km_at_90
experiment                                                  
1b: ±2          0.51     0.86          44.00            3.93
2: ±2 + tilt    0.51     0.85          44.33            3.23


In [13]:
def run_seeds(name, features, data, seeds=[1, 2, 3], leaf=1):
    rows = []
    for seed in seeds:
        rows.append(run_experiment(name, features, data, seed, leaf))
    result = pd.DataFrame(rows).drop(columns=["experiment"]).mean()
    result["experiment"] = name
    return result.to_dict()

**Result of experiment 2 (± 2 neighbours + pitch tilt):** the car's average tilt compared to the drive's normal tilt, for the window and its ±2 neighbours (nose up when driving onto a heave, nose down when leaving it).

A single run looked like a trade-off (2 fewer bumps found, fewer false alarm spots), so both versions were repeated with 3 random seeds:

| | Bumps found (avg) | False alarm spots/km at ≥ 90% found (avg) |
|---|---|---|
| 1b: ±2 neighbours | 44.0 | 3.93 |
| 2: ±2 + tilt | 44.3 | **3.23** |

The tilt gave fewer false alarm spots for every seed, with the same number of bumps found. **Decision: keep the tilt.** Experiment 2 is the new best version.

**Lesson:** single runs vary by 1–3 bumps depending on the random seed. From now on, experiments are compared by their average over 3 seeds.

### Experiment 3: RBCSV_80

The workshop and a classmate pointed to RBCSV_80 as a strong predictor. In 05 it lowered window F1 (more false alarms on the rougher Kaittiaisentie). Here it is tested again: relative to each drive's normal level, with and without ±2 neighbours, judged at bump level and averaged over 3 seeds. Only RBCSV_80 is used, because the Jeremy's starter notebooks drop RBCSV_60 as a data leak.

In [14]:
# Relative RBCSV_80: mean and peak in the window, divided by the drive's calm level
rbcsv_cols = ["RBCSV80_mean", "RBCSV80_max"]
for col in rbcsv_cols:
    calm = all_windows[kept].groupby("drive")[col].quantile(0.25)
    all_windows[col + "_rel"] = all_windows[col] / all_windows["drive"].map(calm)

rbcsv_rel = ["RBCSV80_mean_rel", "RBCSV80_max_rel"]

# Their neighbours, ±2 windows
for shift, name in [(1, "_prev"), (-1, "_next"), (2, "_prev2"), (-2, "_next2")]:
    all_windows = add_neighbours(all_windows, rbcsv_rel, shift, name)

rbcsv_neighbours = []
for col in rbcsv_rel:
    for name in ["_prev", "_next", "_prev2", "_next2"]:
        rbcsv_neighbours.append(col + name)

print(len(rbcsv_rel), "own +", len(rbcsv_neighbours), "neighbour features")

2 own + 8 neighbour features


In [15]:
windows = all_windows[kept].copy()
windows["is_bump"] = (windows["label"] == "bump").astype(int)

best_features = neighbour2_features + tilt_features

log3 = []
log3.append(run_seeds("2: ±2 + tilt (current best)", best_features, windows))
log3.append(run_seeds("3a: + RBCSV80", best_features + rbcsv_rel, windows))
log3.append(run_seeds("3b: + RBCSV80 with ±2 neighbours", best_features + rbcsv_rel + rbcsv_neighbours, windows))

columns = ["experiment", "avg_f1", "avg_auc", "found_default", "spots_km_default", "spots_km_at_90"]
print(pd.DataFrame(log3)[columns].round(2).to_string(index=False))

                      experiment  avg_f1  avg_auc  found_default  spots_km_default  spots_km_at_90
     2: ±2 + tilt (current best)    0.51     0.85          44.33              3.23            3.23
                   3a: + RBCSV80    0.51     0.85          44.67              3.62            3.62
3b: + RBCSV80 with ±2 neighbours    0.51     0.85          43.33              2.96            3.27


**Result of experiment 3 (RBCSV_80, 3-seed averages):**

| | Bumps found | Spots/km at ≥ 90% found |
|---|---|---|
| 2: ±2 + tilt (current best) | 44.3 | 3.23 |
| 3a: + RBCSV_80 (relative mean, peak) | 44.7 | 3.62 |
| 3b: + RBCSV_80 with ±2 neighbours | 43.3 | 3.27 |

RBCSV_80 does not improve the model: either more false alarm spots (3a) or fewer bumps found (3b); window F1 and AUC unchanged. With context from neighbouring windows, the sensor features already carry what RBCSV adds. **Decision: not used.** The final model uses only the motion sensors.

### Experiment 4: RF setting `min_samples_leaf`

Minimum number of windows in each leaf of a tree. Higher values = more general rules, less memorising of single windows.

In [16]:
log4 = []
for leaf in [1, 5, 10, 20]:
    log4.append(run_seeds("4: leaf " + str(leaf), best_features, windows, leaf=leaf))

print(pd.DataFrame(log4)[columns].round(2).to_string(index=False))

experiment  avg_f1  avg_auc  found_default  spots_km_default  spots_km_at_90
 4: leaf 1    0.51     0.85          44.33              3.23            3.23
 4: leaf 5    0.56     0.86          46.00              4.01            2.71
4: leaf 10    0.54     0.86          46.00              4.91            2.98
4: leaf 20    0.52     0.86          47.00              6.77            2.83


**Result of experiment 4 (`min_samples_leaf`, 3-seed averages):**

| Leaf | Window F1 | Bumps found (default) | Spots/km at ≥ 90% found |
|---|---|---|---|
| 1 (default) | 0.51 | 44.3 | 3.23 |
| **5** | **0.56** | **46.0** | **2.71** |
| 10 | 0.54 | 46.0 | 2.98 |
| 20 | 0.52 | 47.0 | 2.83 |

With at least 5 windows per leaf, the trees learn more general rules instead of memorising single windows. **Decision: `min_samples_leaf = 5`.** Best version: ±2 neighbours + pitch tilt + leaf 5.

In [17]:
print(pd.DataFrame(log4)[["experiment", "threshold_at_90"]].round(2).to_string(index=False))

experiment  threshold_at_90
 4: leaf 1             0.50
 4: leaf 5             0.58
4: leaf 10             0.60
4: leaf 20             0.63


### Fair threshold: chosen without the test road

So far, the threshold for "≥ 90% found" was chosen by looking at the test roads' results (fine for comparing experiments, but slightly optimistic as a final result). Here, for each test road, the threshold is chosen using only the two training roads ("practice predictions": train on one, predict the other, and swap), and then applied to the test road. The test road never influences the threshold, so this is an honest estimate for a new road.

* First, threshold in simple manner means telling the model that this is the line, if the values are above this consider it a bump.

* Second we are experimenting this cause we know the threshold for the dataset that we got and the predictions works better with it. But what if we get the dataset which we will, that we don't know in advance at what threshold the model will perform better. So, just experimenting how bad it can go.

In [18]:
def train_predict(train, test, features, seed, leaf):
    model = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   random_state=seed, min_samples_leaf=leaf)
    model.fit(train[features], train["is_bump"])
    p = test[["drive", "road", "window", "is_bump"]].copy()
    p["prob"] = model.predict_proba(test[features])[:, 1]
    return p


def pick_threshold(p, goal=0.9):
    best_spots = None
    best_t = None
    for t in np.arange(0.05, 1.0, 0.05):
        p["pred"] = (p["prob"] >= t).astype(int)
        b = bump_level(p)
        if b["detection_rate"] >= goal:
            if best_spots is None or b["spots_per_km"] < best_spots:
                best_spots = b["spots_per_km"]
                best_t = t
    if best_t is None:
        best_t = 0.05
    return best_t

In [19]:
def fair_test(features, data, seed, leaf=5):
    all_p = []
    thresholds = {}

    for test_road in data["road"].unique():
        train = data[data["road"] != test_road]
        test = data[data["road"] == test_road]

        # 1. Practice predictions, using only the two training roads
        practice = []
        for practice_road in train["road"].unique():
            practice_train = train[train["road"] != practice_road]
            practice_test = train[train["road"] == practice_road]
            practice.append(train_predict(practice_train, practice_test, features, seed, leaf))

        # 2. Choose the threshold on the practice predictions
        t = pick_threshold(pd.concat(practice))
        thresholds[test_road] = round(t, 2)

        # 3. Train on both training roads, apply that threshold to the test road
        p = train_predict(train, test, features, seed, leaf)
        p["pred"] = (p["prob"] >= t).astype(int)
        all_p.append(p)

    result = bump_level(pd.concat(all_p))
    result["thresholds"] = thresholds
    return result

In [20]:
fair_rows = []
for seed in [1, 2, 3]:
    r = fair_test(best_features, windows, seed, leaf=5)
    print("seed", seed, "| thresholds:", r["thresholds"])
    fair_rows.append(r)

fair = pd.DataFrame(fair_rows)
print()
print(fair[["bumps_found", "bumps_total", "detection_rate", "spots_per_km"]].round(2).to_string(index=False))
print("average: found", round(fair["bumps_found"].mean(), 1), "of 48 | spots/km", round(fair["spots_per_km"].mean(), 2))

seed 1 | thresholds: {'Iisingintie': np.float64(0.15), 'Kaittiaisentie': np.float64(0.05), 'sinetänsalmentie': np.float64(0.6)}
seed 2 | thresholds: {'Iisingintie': np.float64(0.1), 'Kaittiaisentie': np.float64(0.05), 'sinetänsalmentie': np.float64(0.55)}
seed 3 | thresholds: {'Iisingintie': np.float64(0.15), 'Kaittiaisentie': np.float64(0.05), 'sinetänsalmentie': np.float64(0.6)}

 bumps_found  bumps_total  detection_rate  spots_per_km
          48           48            1.00         11.22
          48           48            1.00          8.58
          47           48            0.98         10.92
average: found 47.7 of 48 | spots/km 10.24


In [21]:
base_rows = []
for seed in [1, 2, 3]:
    base_rows.append(fair_test(rel_features, windows, seed, leaf=1))

base = pd.DataFrame(base_rows)
print("BEFORE tuning: found", round(base["bumps_found"].mean(), 1), "of 48 | spots/km", round(base["spots_per_km"].mean(), 2))
print("AFTER tuning:  found", round(fair["bumps_found"].mean(), 1), "of 48 | spots/km", round(fair["spots_per_km"].mean(), 2))

BEFORE tuning: found 48.0 of 48 | spots/km 16.72
AFTER tuning:  found 47.7 of 48 | spots/km 10.24


## Conclusion of 10

**Final model:** random forest (300 trees, `class_weight="balanced"`, `min_samples_leaf=5`) on:
- the 7 relative sensor features (std ÷ the drive's calm level) of the window,
- the same features of the windows ±1 and ±2 away (25 m of context),
- the pitch tilt (average tilt vs the drive's normal tilt) of the window and its ±2 neighbours.

**Tuning steps (road split, bump level, 3-seed averages from experiment 2 on):**

| Step | Result | Kept |
|---|---|---|
| Neighbours ±1 / ±2 | F1 0.44 → 0.51, AUC 0.76 → 0.86, fewer false alarm spots | ±2 YES |
| Neighbours ±3 | short bumps lost (42/48 found) | NO |
| Pitch tilt | false alarm spots 3.93 → 3.23 per km, for every seed | YES |
| RBCSV_80 | no improvement | NO |
| Gap filling | not needed: neighbours already fill heave tops | NO |
| `min_samples_leaf = 5` | F1 0.56, spots 3.23 → 2.71 per km | YES |

**New metric:** false alarm *spots* (neighbouring alarm windows = one place to check on the road), added because neighbour features make alarms come in groups.

**Fair threshold:** choosing the threshold with the test results is optimistic. With a strict method (threshold chosen only from the training roads):

| | Bumps found | False alarm spots/km |
|---|---|---|
| Before tuning | 48 of 48 | 16.7 |
| **After tuning** | **47.7 of 48** | **10.2** (39% fewer) |

- The tuning improves the model also under the strict evaluation.
- With only 3 roads, the threshold cannot be chosen reliably in advance: the "practice" models trained on a single road are too weak, so they suggest thresholds that are too sensitive. On a new road, performance lies between the strict result (10.2 spots/km at 99% found) and the optimistic one (2.7 spots/km at ≥ 90% found), depending on the threshold.
- **The final model uses the threshold chosen on all three roads (≈ 0.55–0.6): ≥ 90% of the bumps found with about 2.7 false alarm spots per km.** The strict result (10.2 spots/km) is the worst case if the threshold has to be chosen blindly; the dashboard therefore gets a sensitivity slider so the user can adjust the threshold for their road.

---------------------------------------------------------------------------------------------------------

## Saving the final model
1. Cross-validated predictions (each road predicted by a model trained on the other two): for the dashboard demo.
2. The final model trained on all three roads, with its feature list and default threshold: for new drives.

In [22]:
import joblib
from pathlib import Path
Path("models").mkdir(exist_ok=True)

# ---- 1. Cross-validated predictions (each road predicted by a model that never saw it) ----
cv_parts = []
for test_road in windows["road"].unique():
    train = windows[windows["road"] != test_road]
    test = windows[windows["road"] == test_road]
    # Train a model on the two training roads, and predict the test road.
    # Random seed=101, min_samples_leaf=5 (as in the best model from the previous experiments)
    cv_parts.append(train_predict(train, test, best_features, 101, 5))
cv_pred = pd.concat(cv_parts)

# The default threshold: fewest false alarm spots while finding at least 90%
final_threshold = pick_threshold(cv_pred)
cv_pred["pred"] = (cv_pred["prob"] >= final_threshold).astype(int)
cv_pred.to_csv("predictions/predictions_rf_final.csv", index=False)

print("default threshold:", round(final_threshold, 2))
print(bump_level(cv_pred))

# ---- 2. The final model, trained on all three roads ----
final_model = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                     random_state=101, min_samples_leaf=5)
final_model.fit(windows[best_features], windows["is_bump"])

joblib.dump({"model": final_model, "features": best_features, "threshold": final_threshold},
            "models/rf_final.joblib")
print("saved: models/rf_final.joblib")

default threshold: 0.6
{'bumps_total': 48, 'bumps_found': 44, 'detection_rate': 0.9166666666666666, 'real_false_alarms': 95, 'false_alarms_per_km': 6.964809384164223, 'spots_per_km': 2.4193548387096775}
saved: models/rf_final.joblib
